<h1>LLaVA Inference Testing</h1>

In [27]:
import lmstudio as lms

with lms.Client() as client:
    model = client.llm.model("llava-llama-3-8b-v1_1")
    result = model.respond("Say 'Hello World'")
    print(result)

Hello, world!


In [28]:
from pycocotools.coco import COCO
import random
import os

image_dir = "data/val2017"
caption_dir = "data/captions_val2017.json"
instance_dir = "data/instances_val2017.json"

coco_caps = COCO(caption_dir)
coco_inst = COCO(instance_dir)

img_ids = list(coco_inst.imgs.keys())

random.seed(42)
img_ids = random.sample(img_ids, 200)

dataset = []

for img_id in img_ids:

    img_info = coco_inst.loadImgs(img_id)[0]
    file_name = img_info["file_name"]
    image_path = os.path.join(image_dir, file_name)

    # captions
    cap_ids = coco_caps.getAnnIds(imgIds=img_id)
    caps = coco_caps.loadAnns(cap_ids)
    captions = [c["caption"] for c in caps]

    # boxes
    ann_ids = coco_inst.getAnnIds(imgIds=img_id)
    anns = coco_inst.loadAnns(ann_ids)

    objects = []
    for a in anns:
        cat_name = coco_inst.loadCats(a["category_id"])[0]["name"]
        objects.append({
            "category": cat_name,
            "bbox": a["bbox"]
        })

    dataset.append({
        "image_id": img_id,
        "image_path": image_path,
        "captions": captions,
        "objects": objects
    })

print(dataset[0]["image_path"])
print(dataset[0]["captions"][:2])
print(dataset[0]["objects"][:2])

loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
loading annotations into memory...
Done (t=0.23s)
creating index...
index created!
data/val2017\000000301061.jpg
['A person is moving green hay towards an elephant that is inside the back of a white truck.', 'A man pulls an elephant out of a truck.']
[{'category': 'person', 'bbox': [5.75, 284.32, 66.16, 316.41]}, {'category': 'elephant', 'bbox': [128.29, 155.95, 196.63, 252.64]}]


In [29]:
llava_responses = []

with lms.Client() as client:
    model = client.llm.model("llava-llama-3-8b-v1_1")

    for sample in dataset:
        image_path = sample["image_path"]
        image_handle = lms.prepare_image(image_path)
        chat = lms.Chat()
        chat.add_user_message("Describe this image please", images=[image_handle])
        response = model.respond(chat)

        print(response)
        llava_responses.append(response)

In the heart of a bustling city, an unusual scene unfolds. A large truck, its white body gleaming under the sunlight, is parked on a street lined with trees and buildings. The truck's cargo area is open, revealing a pile of hay that seems to have caught the attention of a curious elephant.

The elephant, with its majestic gray skin, stands tall in the back of the truck. Its trunk, a symbol of strength and dexterity, is extended towards the ground, reaching out for the hay as if it were a feast laid out before it. The elephant's position in the truck bed suggests it has found an unexpected source of sustenance amidst the urban landscape.

In the foreground, a person stands observing this unique interaction. Their presence adds a human element to the scene, perhaps indicating their role in this unusual encounter. The image captures a moment where nature and city life intersect, creating a snapshot that is both ordinary and extraordinary at once.
In the heart of a bustling city, a daring 

In [30]:
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from PIL import Image

blip_responses = []

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b", device_map={"": 0}, dtype=torch.float16)

for sample in dataset:
    prompt = "Question: Describe this image please. Answer:"
    inputs = processor(images=[Image.open(sample["image_path"]).convert("RGB")], text=prompt, return_tensors="pt").to(device, torch.float16)
    response_ids = model.generate(**inputs)
    response = processor.batch_decode(response_ids, skip_special_tokens=True)[0].strip()

    print(response)

Loading weights: 100%|██████████| 1247/1247 [00:06<00:00, 207.33it/s, Materializing param=vision_model.post_layernorm.weight]                               


Question: Describe this image please. Answer: An elephant is being loaded into a truck
Question: Describe this image please. Answer: A skateboarder is doing a trick on a street
Question: Describe this image please. Answer: A tennis player on a tennis court
Question: Describe this image please. Answer: I am a man
Question: Describe this image please. Answer: A woman's hair is on a table
Question: Describe this image please. Answer: A table with wine glasses, bread, and a bottle of wine
Question: Describe this image please. Answer: A living room with a sofa, table, and a television
Question: Describe this image please. Answer: A man sitting at a table in a train station
Question: Describe this image please. Answer: A computer desk with two monitors and a keyboard and mouse
Question: Describe this image please. Answer: A mother elephant and her baby elephant
Question: Describe this image please. Answer: A toilet with a colorful seat cover
Question: Describe this image please. Answer: A pl

In [ ]:
with lms.Client() as client:
    model = client.llm.model("openai/gpt-oss-20b")

    for sample in dataset:
        image_path = sample.filepath
        image_handle = lms.prepare_image(image_path)
        chat = lms.Chat()
        chat.add_user_message("Given the ground-truth qualities of an image including captions" + )
        response = model.respond(chat)

        print(response)
        llava_responses.append(response)